In [ ]:
import os
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
import csv
import pathlib
from typing import Set


In [ ]:
# Check the device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

# Set the desired dtype
torch_dtype = torch.float16
print(torch.empty(1, dtype=torch_dtype).dtype)  # This will display "torch.float16"

In [ ]:
# Load model and processor
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Florence-2-large", 
    torch_dtype=torch_dtype, 
    trust_remote_code=True
).to(device)

processor = AutoProcessor.from_pretrained(
    "microsoft/Florence-2-large", 
    trust_remote_code=True
)  

In [ ]:
import os
import csv
import pathlib
from typing import Set, Dict, Tuple

def analyze_caption_mismatches(image_dir: str, output_file: str) -> Tuple[Set[str], Set[str], Dict[str, int]]:
    """
    Analyzes mismatches between image files and caption entries.
    Returns:
        - orphaned_captions: captions that exist but have no corresponding image
        - missing_captions: images that exist but have no caption
        - duplicate_captions: dictionary of filenames and their count in CSV
    """
    # Get all image files
    valid_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp'}
    dir_path = pathlib.Path(image_dir)
    image_files = set()
    
    for file_path in dir_path.iterdir():
        if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
            image_files.add(file_path.name)
            
    # Read CSV and track duplicates
    existing_images = set()
    duplicate_counter: Dict[str, int] = {}
    
    if pathlib.Path(output_file).exists():
        with open(output_file, mode='r', newline='', encoding='utf-8') as csvfile:
            csv_reader = csv.reader(csvfile)
            next(csv_reader, None)  # Skip header
            for row in csv_reader:
                if row:
                    filename = row[0]
                    # Count occurrences
                    duplicate_counter[filename] = duplicate_counter.get(filename, 0) + 1
                    existing_images.add(filename)
    
    # Calculate mismatches
    missing_captions = image_files - existing_images
    orphaned_captions = existing_images - image_files
    
    # Only keep entries that appear more than once in duplicate_counter
    duplicate_captions = {k: v for k, v in duplicate_counter.items() if v > 1}
    
    # Print analysis
    print(f"\nAnalysis Results:")
    print(f"Total images in directory: {len(image_files)}")
    print(f"Total entries in CSV: {sum(duplicate_counter.values())}")
    print(f"Unique entries in CSV: {len(existing_images)}")
    
    if orphaned_captions:
        print(f"\nOrphaned captions (in CSV but no image file): {len(orphaned_captions)}")
        print("First few examples:")
        for filename in list(orphaned_captions)[:5]:
            print(f"  - {filename}")
            
    if duplicate_captions:
        print(f"\nDuplicate entries in CSV:")
        for filename, count in list(duplicate_captions.items())[:5]:
            print(f"  - {filename}: {count} times")
            
    if missing_captions:
        print(f"\nImages missing captions: {len(missing_captions)}")
        print("First few examples:")
        for filename in list(missing_captions)[:5]:
            print(f"  - {filename}")
            
    return orphaned_captions, missing_captions, duplicate_captions

if __name__ == "__main__":
    # Configuration
    IMAGE_DIR = "images_transformed"
    OUTPUT_FILE = "results/captions.csv"
    
    # Run analysis
    orphaned, missing, duplicates = analyze_caption_mismatches(IMAGE_DIR, OUTPUT_FILE)

In [ ]:
 # Iterate over all images in the directory
for image_file in os.listdir(image_dir):
    if image_file in existing_images:
        print(f"Skipping {image_file}, already processed.")
        continue

    image_path = os.path.join(image_dir, image_file)
    try:
        # Open image
        image = Image.open(image_path).convert("RGB")

        # Process inputs
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(device, torch_dtype)

        # Generate output
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=4096,
            num_beams=3,
            do_sample=False
        )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
        detailed_caption = generated_text  # Assuming post_process_generation is not available

        # Write the result to the CSV
        with open(output_file, mode="a", newline="", encoding="utf-8") as csvfile:
            csv_writer = csv.writer(csvfile)
            csv_writer.writerow([image_file, detailed_caption])
            csvfile.flush()  # Ensure data is written to file
        print(f"Processed: {image_file}, Detailed Caption: {detailed_caption}")

    except Exception as e:
        print(f"Error processing {image_file}: {e}")

print(f"Results saved to {output_file}")